# 04 — Correctness checks, diagnostics, and defensible claims

**Goal:** verify solver and replay invariants, compare the data with the paper's published
advertiser counts, show model/allocation diagnostics, and generate result wording directly
from executed tables.


In [ ]:
from pathlib import Path
import os

# Run correctly whether Jupyter starts in the project root or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
ARTIFACT_ROOT = Path(os.getenv("IPINYOU_ARTIFACT_ROOT", PROJECT_ROOT / "artifacts"))
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")


In [ ]:
from pathlib import Path
from itertools import product
import hashlib
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:  # Allows the script-style smoke test to run outside Jupyter.
    class Markdown(str):
        pass

    def display(*objects):
        for obj in objects:
            print(obj)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    median_absolute_error,
    roc_auc_score,
)
from sklearn.model_selection import ParameterSampler

import lightgbm as lgb
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy import optimize, sparse

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)
pd.set_option("display.max_colwidth", None)

RANDOM_STATE = 42
NOTEBOOK_SCHEMA_VERSION = "2.2"
TARGET_ADVERTISERS = [1458, 2997]

# Data resolution
USE_KAGGLEHUB = False
KAGGLE_DATASET_SLUG = "pleaseholdme/ipinyou"
DATA_ROOT_OVERRIDE = os.getenv("IPINYOU_DATA_ROOT")

# Eligibility thresholds are data-quality gates, not statistical guarantees.
MIN_TRAIN_ROWS = 50_000
MIN_TRAIN_CLICKS = 100
MIN_TEST_ROWS = 20_000
MIN_TEST_CLICKS = 20

# Outer chronological development split.
FIT_FRAC = 0.70
CAL_FRAC = 0.15

# Hyperparameter selection happens only inside FIT_FRAC.
TUNING_WINDOW_MAX_ROWS = 600_000
TUNING_EVAL_FRAC = 0.20
CTR_TUNING_CANDIDATES = 8
COST_TUNING_CANDIDATES = 6
EARLY_STOPPING_ROUNDS = 60

# Planning and uncertainty specification.
PCTR_BINS = 10
COST_BINS = 5
UPPER_COST_TARGET_COVERAGE = 0.80
PRIMARY_BUDGET_FRACTION = 0.50
BUDGET_FRACTIONS = [0.20, 0.35, 0.50, 0.65, 0.80]

# Solver controls.
SOLVER_TIME_LIMIT_SEC = 30
SOLVER_MIP_REL_GAP = 1e-7
MAX_ACCEPTABLE_REPORTED_MIP_GAP = 1e-3

# Temporal and execution policy.
STRICT_TEMPORAL_HOLDOUT = True
ROUND_OVERLAP_CUTOFF_TO_NEXT_HOUR = True
REQUIRE_BID_TO_CLEAR = True
REQUIRE_FLOOR_TO_CLEAR = True

SMOKE_TEST = os.getenv("IPINYOU_SMOKE_TEST", "0") == "1"
if SMOKE_TEST:
    MIN_TRAIN_ROWS = 1_000
    MIN_TRAIN_CLICKS = 10
    MIN_TEST_ROWS = 500
    MIN_TEST_CLICKS = 5
    BUDGET_FRACTIONS = [0.35, 0.50, 0.65]
    TUNING_WINDOW_MAX_ROWS = 4_000
    CTR_TUNING_CANDIDATES = 2
    COST_TUNING_CANDIDATES = 2

config = pd.Series(
    {
        "schema_version": NOTEBOOK_SCHEMA_VERSION,
        "smoke_test": SMOKE_TEST,
        "target_advertisers": TARGET_ADVERTISERS,
        "fit_fraction": FIT_FRAC,
        "calibration_fraction": CAL_FRAC,
        "tuning_window_max_rows": TUNING_WINDOW_MAX_ROWS,
        "ctr_tuning_candidates": CTR_TUNING_CANDIDATES,
        "cost_tuning_candidates": COST_TUNING_CANDIDATES,
        "budget_fractions": BUDGET_FRACTIONS,
        "primary_budget_fraction": PRIMARY_BUDGET_FRACTION,
        "upper_cost_target_coverage": UPPER_COST_TARGET_COVERAGE,
        "bid_must_clear_payprice": REQUIRE_BID_TO_CLEAR,
        "bid_must_clear_slot_floor": REQUIRE_FLOOR_TO_CLEAR,
        "scipy_milp_available": hasattr(optimize, "milp"),
        "lightgbm_version": lgb.__version__,
    },
    name="value",
)
display(config.to_frame())


In [ ]:
import joblib

PREPARED_DIR = ARTIFACT_ROOT / "01_prepared"
MODEL_DIR = ARTIFACT_ROOT / "02_models_and_scores"
POLICY_DIR = ARTIFACT_ROOT / "03_policy_replay"
for required in [
    PREPARED_DIR / "data_manifest.json",
    MODEL_DIR / "model_manifest.json",
    POLICY_DIR / "primary_results.csv",
]:
    if not required.exists():
        raise FileNotFoundError(f"Missing {required}; run notebooks 01–03 in order.")

data_manifest = json.loads((PREPARED_DIR / "data_manifest.json").read_text())
model_manifest = json.loads((MODEL_DIR / "model_manifest.json").read_text())
ELIGIBLE_ADVERTISERS = [int(x) for x in data_manifest["eligible_advertisers"]]
DATA_ROOT = Path(data_manifest["data_root"])
results = pd.read_csv(POLICY_DIR / "primary_results.csv")
model_metrics = pd.read_csv(MODEL_DIR / "model_metrics.csv")
audit_df = pd.read_csv(PREPARED_DIR / "data_audit.csv")
paper_budget_results = pd.read_csv(POLICY_DIR / "paper_budget_sensitivity.csv")

GROUP_COLS = ["hour_key", "adexchange_key", "pctr_bucket", "cost_bucket"]
CAMPAIGN_RUNS = {}
for advertiser in ELIGIBLE_ADVERTISERS:
    CAMPAIGN_RUNS[advertiser] = {
        "replays": joblib.load(
            POLICY_DIR / f"advertiser_{advertiser}_primary_replays.joblib"
        ),
        "test_scored": pd.read_parquet(
            MODEL_DIR / f"advertiser_{advertiser}_test_scored.parquet"
        ),
        "planning": pd.read_parquet(
            MODEL_DIR / f"advertiser_{advertiser}_planning.parquet"
        ),
        "ctr_search": pd.read_csv(
            MODEL_DIR / f"advertiser_{advertiser}_ctr_search.csv"
        ),
        "cost_search": pd.read_csv(
            MODEL_DIR / f"advertiser_{advertiser}_cost_search.csv"
        ),
        "test_calibration_table": pd.read_csv(
            MODEL_DIR / f"advertiser_{advertiser}_test_calibration.csv"
        ),
    }
print(f"Loaded completed runs for {ELIGIBLE_ADVERTISERS}")


## Dataset alignment with Zhang et al. Table 3

Matching impression counts identify the support available in this mirror. Click and strict-
test differences are displayed rather than silently forced to match the benchmark.


In [ ]:
paper_counts = pd.DataFrame(
    [
        {"advertiser": 1458, "paper_train_impressions": 3_083_056,
         "paper_train_clicks": 2_454, "paper_test_impressions": 614_638,
         "paper_test_clicks": 543},
        {"advertiser": 2997, "paper_train_impressions": 312_437,
         "paper_train_clicks": 1_386, "paper_test_impressions": 156_063,
         "paper_test_clicks": 533},
    ]
)
paper_alignment = audit_df.merge(paper_counts, on="advertiser", how="left")
paper_alignment["train_impression_count_matches_paper"] = (
    paper_alignment.source_train_rows == paper_alignment.paper_train_impressions
)
paper_alignment["test_impression_count_matches_paper"] = (
    paper_alignment.source_test_rows == paper_alignment.paper_test_impressions
)
display(paper_alignment[[
    "advertiser", "source_train_rows", "paper_train_impressions",
    "source_train_clicks", "paper_train_clicks",
    "source_test_rows", "paper_test_impressions",
    "source_test_clicks", "paper_test_clicks",
    "strict_test_rows", "strict_test_clicks",
    "train_impression_count_matches_paper", "test_impression_count_matches_paper",
]])

display(Markdown(
    "The source row counts matching Table 3's impression counts confirm that this mirror is "
    "not the paper's complete bid-request stream. Advertiser 2997's strict-test counts are "
    "smaller because overlapping 26 October rows are removed; this is stricter than the paper."
))


## Solver definitions required for independent correctness tests

Notebook 04 reloads the small solver function so the brute-force test does not rely on
hidden kernel state from Notebook 03.


In [ ]:
def prepare_hour_weights(planning, cost_column):
    weighted = planning.copy()
    weighted["forecast_spend"] = weighted[cost_column] * weighted["forecast_capacity"]
    return weighted.groupby("hour_key", observed=True).forecast_spend.sum().astype(float).to_dict()


def solve_hour_ilp(
    planning,
    hour_key,
    budget,
    cost_column="central_cost",
    value_column="expected_ctr",
    time_limit=SOLVER_TIME_LIMIT_SEC,
):
    hour_plan = planning.loc[
        planning.hour_key.astype(str) == str(hour_key)
    ].reset_index(drop=True).copy()
    if hour_plan.empty or budget <= 0:
        return {
            "status": "no_inventory", "message": "No planning groups or no budget",
            "budget": float(budget), "cost_column": cost_column,
            "value_column": value_column, "mip_gap": 0.0, "allocation": None,
            "quota_cost": 0.0, "quota_impressions": 0, "quota_value": 0.0,
            "feasibility_violation": 0.0,
        }
    n_groups = len(hour_plan)
    objective = -hour_plan[value_column].astype(float).to_numpy()
    lower = np.zeros(n_groups)
    upper = hour_plan.forecast_capacity.astype(float).to_numpy()
    integrality = np.ones(n_groups, dtype=int)
    costs = hour_plan[cost_column].astype(float).to_numpy()
    constraint = optimize.LinearConstraint(
        sparse.csr_matrix(costs.reshape(1, -1)),
        np.array([-np.inf]), np.array([float(budget)]),
    )
    result = optimize.milp(
        c=objective,
        integrality=integrality,
        bounds=optimize.Bounds(lower, upper),
        constraints=constraint,
        options={"time_limit": time_limit, "mip_rel_gap": SOLVER_MIP_REL_GAP},
    )
    status_map = {0: "optimal", 1: "limit_reached", 2: "infeasible", 3: "unbounded", 4: "other_failure"}
    status = status_map.get(result.status, f"status_{result.status}")
    output = {
        "status": status, "message": result.message, "budget": float(budget),
        "cost_column": cost_column, "value_column": value_column,
        "mip_gap": getattr(result, "mip_gap", np.nan), "allocation": None,
        "quota_cost": np.nan, "quota_impressions": 0, "quota_value": np.nan,
        "feasibility_violation": np.nan,
    }
    if status != "optimal" or result.x is None:
        return output
    allocation_vector = np.rint(np.clip(result.x, 0, upper)).astype(int)
    quota_cost = float(costs @ allocation_vector)
    violation = max(0.0, quota_cost - float(budget))
    assert np.all(allocation_vector >= 0) and np.all(allocation_vector <= upper + 1e-9)
    assert violation <= 1e-5 * max(1.0, float(budget))
    allocation = hour_plan[
        ["group_id"] + GROUP_COLS
        + ["forecast_capacity", "expected_ctr", "central_cost", "upper_cost"]
    ].copy()
    allocation["quota_impressions"] = allocation_vector
    allocation = allocation[allocation.quota_impressions > 0].copy()
    output.update(
        {
            "allocation": allocation,
            "quota_cost": quota_cost,
            "quota_impressions": int(allocation_vector.sum()),
            "quota_value": float(np.sum(hour_plan[value_column].to_numpy() * allocation_vector)),
            "feasibility_violation": violation,
        }
    )
    return output


def quota_from_plan(plan):
    if plan["allocation"] is None:
        return {}
    return {
        (str(row.hour_key), str(row.adexchange_key), int(row.pctr_bucket), int(row.cost_bucket)):
        int(row.quota_impressions)
        for row in plan["allocation"].itertuples()
    }


## 13. Correctness, bid-rule, and solver checks

The solver is compared with exhaustive enumeration on deterministic small instances. Additional
assertions verify chronology, solver status, MIP gap, predicted-budget feasibility, bid clearing,
and realized budget feasibility.


In [ ]:
def brute_force_ilp_self_test(n_cases=25):
    rng = np.random.default_rng(2026)
    for case in range(n_cases):
        n_groups = int(rng.integers(2, 5))
        capacity = rng.integers(1, 5, size=n_groups)
        costs = rng.integers(1, 12, size=n_groups).astype(float)
        values = rng.uniform(0.05, 1.0, size=n_groups)
        budget = float(rng.integers(4, max(5, int((costs * capacity).sum()))))
        miniature = pd.DataFrame(
            {
                "group_id": np.arange(n_groups), "hour_key": ["7"] * n_groups,
                "adexchange_key": ["1"] * n_groups,
                "pctr_bucket": np.arange(n_groups), "cost_bucket": np.arange(n_groups),
                "forecast_capacity": capacity, "expected_ctr": values,
                "central_cost": costs, "upper_cost": costs, "unit_value": 1.0,
            }
        )
        solution = solve_hour_ilp(
            miniature, "7", budget, "central_cost", "expected_ctr"
        )
        assert solution["status"] == "optimal"
        best = -np.inf
        for allocation in product(*[range(int(upper) + 1) for upper in capacity]):
            allocation = np.asarray(allocation)
            if costs @ allocation <= budget + 1e-9:
                best = max(best, float(values @ allocation))
        assert abs(solution["quota_value"] - best) <= 1e-7, (case, solution["quota_value"], best)
    return True


print("MILP brute-force correctness self-test:", brute_force_ilp_self_test())
assert (results.scenario_budget_native > 0).all()
assert results.all_solver_optimal.all(), "At least one hourly solve was not optimal."
assert results.max_feasibility_violation.max() <= 1e-5 * results.scenario_budget_native.max()
assert results.max_mip_gap.max() <= MAX_ACCEPTABLE_REPORTED_MIP_GAP
assert results.budget_overrun_pct.max() <= 1e-12

for run in CAMPAIGN_RUNS.values():
    for replay in run["replays"].values():
        if replay["accepted_rows"] and REQUIRE_BID_TO_CLEAR:
            accepted = run["test_scored"].loc[replay["accepted_rows"]]
            policy_cost = (
                "upper_cost_pred" if replay["policy"] == "CTR upper-cost" else "central_cost_pred"
            )
            assert (accepted.payprice <= accepted[policy_cost] + 1e-9).all()
            if REQUIRE_FLOOR_TO_CLEAR:
                assert (accepted.slotprice <= accepted[policy_cost] + 1e-9).all()

print("Core temporal, auction, budget, and solver checks: PASS")


# Results

The advertiser-level table is primary. Cross-advertiser medians are intentionally omitted because
the median of two campaigns is only their midpoint and can hide opposite directions.


## 14. Hyperparameter search audit

Search tables are shown separately for every advertiser. The final holdout metrics are not used
to rank configurations. If tuning improves internal validation but not the holdout, that is a
legitimate result rather than a reason to retune on test.


In [ ]:
for advertiser, run in CAMPAIGN_RUNS.items():
    display(Markdown(f"### Advertiser {advertiser}: CTR search"))
    display(run["ctr_search"])
    display(Markdown(f"### Advertiser {advertiser}: cost search"))
    display(run["cost_search"])


## 15. Predictive and cost-bound diagnostics

Average precision is displayed alongside prevalence and AP lift. Absolute ECE is supplemented by
ECE divided by prevalence because very rare outcomes can make an apparently tiny ECE practically
meaningful. Upper-cost coverage must be assessed on validation and strict test, not only on the
calibration sample used to construct the adjustment.


In [ ]:
diagnostic_columns = [
    "advertiser", "ctr_calibration_choice",
    "val_prevalence", "test_prevalence",
    "val_roc_auc", "test_roc_auc",
    "val_average_precision", "test_average_precision",
    "val_ap_lift_vs_prevalence", "test_ap_lift_vs_prevalence",
    "val_log_loss", "test_log_loss",
    "val_ece", "test_ece", "val_ece_over_prevalence", "test_ece_over_prevalence",
    "cost_val_pred_to_actual_spend", "cost_test_pred_to_actual_spend",
    "upper_cost_target_coverage", "upper_cost_validation_coverage", "upper_cost_test_coverage",
    "upper_cost_validation_mean_width", "upper_cost_test_mean_width",
    "aggregate_scale_factor", "conformal_adjustment_native",
    "pctr_psi", "cost_pred_psi", "planning_coefficient_fallback_rate",
    "strict_test_days", "strict_temporal_gap_seconds",
]
display(model_metrics[[column for column in diagnostic_columns if column in model_metrics]])

calibration_tables = pd.concat(
    [run["test_calibration_table"] for run in CAMPAIGN_RUNS.values()], ignore_index=True
)
display(calibration_tables)


## 16. Primary same-cap comparison

The central CTR and volume policies share the same central predicted costs, bid rule, capacity
limits, hourly pacing, and campaign cap. Only the objective changes. The upper-cost policy uses
different group costs and bid caps, so its total-click comparison must be read together with
realized spend, win rate, utilization, and efficiency.

`CPC improvement` and `efficiency lift` are algebraically related and are not independent pieces
of evidence.


In [ ]:
volume = (
    results[results.policy == "Volume benchmark"]
    .set_index(["advertiser", "budget_fraction"])[
        [
            "realized_clicks", "realized_spend_rmb", "realized_cpc_rmb",
            "clicks_per_1000_rmb", "budget_utilization", "auction_win_rate",
        ]
    ]
    .add_prefix("volume_")
)
comparison = results[results.policy != "Volume benchmark"].join(
    volume, on=["advertiser", "budget_fraction"]
)
comparison["click_lift_vs_volume"] = (
    comparison.realized_clicks - comparison.volume_realized_clicks
) / comparison.volume_realized_clicks.replace(0, np.nan)
comparison["cpc_improvement_vs_volume"] = (
    1 - comparison.realized_cpc_rmb / comparison.volume_realized_cpc_rmb
)
comparison["efficiency_lift_vs_volume"] = (
    comparison.clicks_per_1000_rmb / comparison.volume_clicks_per_1000_rmb - 1
)
comparison["utilization_delta_vs_volume"] = (
    comparison.budget_utilization - comparison.volume_budget_utilization
)

primary = comparison[
    np.isclose(comparison.budget_fraction, PRIMARY_BUDGET_FRACTION)
].copy()
primary_columns = [
    "advertiser", "policy", "realized_clicks", "volume_realized_clicks",
    "click_lift_vs_volume", "realized_spend_rmb", "volume_realized_spend_rmb",
    "budget_utilization", "volume_budget_utilization", "auction_win_rate",
    "volume_auction_win_rate", "realized_cpc_rmb", "volume_realized_cpc_rmb",
    "cpc_improvement_vs_volume", "efficiency_lift_vs_volume",
]
display(primary[primary_columns].sort_values(["policy", "advertiser"]))

direction_summary = (
    primary.groupby("policy")
    .agg(
        advertisers=("advertiser", "nunique"),
        positive_click_directions=("click_lift_vs_volume", lambda values: int((values > 0).sum())),
        negative_click_directions=("click_lift_vs_volume", lambda values: int((values < 0).sum())),
        positive_efficiency_directions=("efficiency_lift_vs_volume", lambda values: int((values > 0).sum())),
    )
    .reset_index()
)
display(direction_summary)


In [ ]:
for advertiser, advertiser_frame in comparison.groupby("advertiser"):
    plt.figure(figsize=(8, 4.5))
    for policy, policy_frame in advertiser_frame.groupby("policy"):
        ordered = policy_frame.sort_values("budget_fraction")
        plt.plot(
            ordered.budget_fraction,
            100 * ordered.click_lift_vs_volume,
            marker="o",
            label=policy,
        )
    plt.axhline(0, color="black", linewidth=1)
    plt.xlabel("Train-derived scenario budget fraction")
    plt.ylabel("Replay click lift vs. volume (%)")
    plt.title(f"Advertiser {advertiser}: sensitivity across budget caps")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 17. Budget, capacity, bidding, and solver audit

Low utilization means that the scenario cap was not the dominant constraint. It must be reported,
not hidden by a click-lift percentage. A warning is generated when primary utilization is below
80%. The reoptimized quota-cost sum can exceed the campaign cap because unfilled hourly quotas
are discarded and the remaining realized budget is reallocated; it is not presented as total
committed spend.


In [ ]:
budget_audit = results[
    [
        "advertiser", "budget_fraction", "policy", "scenario_budget_rmb",
        "realized_spend_rmb", "budget_utilization", "predicted_winning_spend_native",
        "reoptimized_quota_cost_sum_native", "bid_attempts", "accepted_impressions",
        "market_price_losses", "floor_losses", "auction_win_rate",
        "budget_overrun_pct", "all_solver_optimal",
        "max_mip_gap", "max_feasibility_violation",
    ]
].sort_values(["advertiser", "budget_fraction", "policy"])
display(budget_audit)

primary_utilization = results[
    np.isclose(results.budget_fraction, PRIMARY_BUDGET_FRACTION)
][["advertiser", "policy", "budget_utilization"]]
low_utilization = primary_utilization[primary_utilization.budget_utilization < 0.80]
if len(low_utilization):
    display(Markdown(
        "**Interpretation warning:** at least one primary policy used less than 80% of its "
        "scenario cap. Treat the cap as non-binding for those rows and interpret click lift "
        "together with realized spend and win rate."
    ))

solver_audit = (
    results.groupby(["advertiser", "policy"])
    .agg(
        all_hours_optimal=("all_solver_optimal", "all"),
        max_mip_gap=("max_mip_gap", "max"),
        max_budget_overrun=("budget_overrun_pct", "max"),
        max_feasibility_violation=("max_feasibility_violation", "max"),
    )
    .reset_index()
)
display(solver_audit)


## 18. Automatically generated result wording

This cell is the source of truth for the README result block. Copy its executed output after a
cleared-kernel real-data run. Do not preserve older hand-written click counts.


In [ ]:
def format_percent(value):
    return "NA" if not np.isfinite(value) else f"{100 * value:+.2f}%"


def generate_primary_result_markdown(primary_frame):
    lines = [
        f"### Primary replay result ({PRIMARY_BUDGET_FRACTION:.0%} train-derived budget cap)",
        "",
        "| Advertiser | Policy | Clicks | Volume clicks | Click lift | Spend (RMB) | "
        "Utilization | CPC improvement | Efficiency lift |",
        "|---:|---|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for row in primary_frame.sort_values(["advertiser", "policy"]).itertuples():
        lines.append(
            f"| {row.advertiser} | {row.policy} | {row.realized_clicks} | "
            f"{row.volume_realized_clicks:.0f} | {format_percent(row.click_lift_vs_volume)} | "
            f"{row.realized_spend_rmb:,.2f} | {row.budget_utilization:.1%} | "
            f"{format_percent(row.cpc_improvement_vs_volume)} | "
            f"{format_percent(row.efficiency_lift_vs_volume)} |"
        )
    lines.extend(["", "Direction counts:"])
    for policy, policy_frame in primary_frame.groupby("policy"):
        positive = int((policy_frame.click_lift_vs_volume > 0).sum())
        lines.append(f"- {policy}: positive click direction in {positive}/{len(policy_frame)} advertisers.")
    lines.extend(
        [
            "",
            "These are advertiser-specific descriptive replay results. They do not establish "
            "population-level, causal, or statistically significant improvement.",
        ]
    )
    return "\n".join(lines)


primary_result_markdown = generate_primary_result_markdown(primary)
display(Markdown(primary_result_markdown))
print("\nREADME_RESULT_BLOCK_START\n")
print(primary_result_markdown)
print("\nREADME_RESULT_BLOCK_END")


## 19. Uncertainty and inference boundary

No confidence interval is reported from one to three holdout days. A day bootstrap over such a
short, adaptively budget-coupled sequence creates false precision because later decisions depend
on earlier realized spend. The notebook therefore reports raw advertiser-level directions and
the number of strict-test days. Formal inference requires substantially more independent temporal
blocks and more advertisers.


## 20. Limitations

1. **Historical support.** Replay cannot reveal outcomes for auctions absent from the released
   opportunity stream.
2. **Static market assumption.** Logged `payprice` is treated as unchanged by the evaluated rule.
3. **Simple bid cap.** The submitted bid is a predicted-price ceiling, not a separately optimized
   value-based production bid.
4. **No causal effect.** pCTR estimates response, not incremental treatment effect.
5. **Only two advertisers.** Results are descriptive advertiser case studies.
6. **Short holdouts.** Few independent days preclude reliable uncertainty intervals.
7. **Experimental budgets.** Train-derived caps are not contractual campaign budgets and may be
   non-binding.
8. **Capacity forecast error.** Historical average segment capacity may differ from future supply.
9. **Marginal upper-cost calibration.** Row-level empirical coverage does not guarantee aggregate
   hourly or campaign spend coverage under dependence.
10. **Planning fallback.** Groups missing from the out-of-fit reference use full-history predicted
    coefficients; the fallback rate is reported.
11. **Dataset vintage and mirror provenance.** Results apply only to the displayed hashes of the
    historical files.
12. **Model-selection scope.** The capped parameter search is not exhaustive, and better internal
    validation does not guarantee better strict-holdout performance.


## 21. Reproducibility record

Execute the notebook from a cleared kernel with `IPINYOU_SMOKE_TEST=0`. Preserve the complete
audit, hashes, tuning tables, diagnostics, primary table, generated README block, and version
record. Never replace mixed or negative advertiser results with pooled totals.


In [ ]:
import platform
import scipy
import sklearn

reproducibility = pd.Series(
    {
        "notebook_schema_version": NOTEBOOK_SCHEMA_VERSION,
        "python": platform.python_version(),
        "pandas": pd.__version__, "numpy": np.__version__,
        "scikit_learn": sklearn.__version__, "scipy": scipy.__version__,
        "lightgbm": lgb.__version__, "random_state": RANDOM_STATE,
        "data_root": str(DATA_ROOT),
        "eligible_advertisers": ",".join(map(str, ELIGIBLE_ADVERTISERS)),
        "budget_fractions": str(BUDGET_FRACTIONS),
        "primary_budget_fraction": PRIMARY_BUDGET_FRACTION,
        "pctr_bins": PCTR_BINS, "cost_bins": COST_BINS,
        "upper_cost_target_coverage": UPPER_COST_TARGET_COVERAGE,
        "strict_temporal_holdout": STRICT_TEMPORAL_HOLDOUT,
        "round_overlap_cutoff_to_next_hour": ROUND_OVERLAP_CUTOFF_TO_NEXT_HOUR,
        "require_bid_to_clear": REQUIRE_BID_TO_CLEAR,
        "require_floor_to_clear": REQUIRE_FLOOR_TO_CLEAR,
    },
    name="value",
)
display(reproducibility.to_frame())


# Appendix — compact formulation

For advertiser \(a\), hour \(h\), and planning group \(g\):

- \(v_{ag}\): calibrated expected click probability;
- \(c_{ag}\): central predicted clearing price;
- \(u_{ag}\ge c_{ag}\): calibrated upper clearing-price coefficient;
- \(N_{ag}\): historical exposure-normalized capacity;
- \(B_{ah}\): adaptively paced current-hour budget.

The central policy solves

\[
\max_x \sum_g v_{ag}x_{agh}
\quad\text{s.t.}\quad
\sum_g c_{ag}x_{agh}\le B_{ah},\quad
0\le x_{agh}\le N_{ag},\quad x_{agh}\in\mathbb Z_+.
\]

The volume benchmark replaces \(v_{ag}\) with 1. The upper-cost policy uses

\[
\mathcal U_{ag}=[c_{ag},u_{ag}],
\qquad
\max_{\tilde c\in\mathcal U_a}\sum_g\tilde c_{ag}x_{agh}\le B_{ah}.
\]

Because \(x\ge0\), the box-robust counterpart is

\[
\sum_g u_{ag}x_{agh}\le B_{ah}.
\]

During replay, the row-level cost coefficient is submitted as the bid cap. The auction is won
only if that cap clears the logged paying price. Only won impressions consume realized budget
and contribute clicks.

### References

- Liao et al., *iPinYou Global RTB Bidding Algorithm Competition Dataset*:
  https://contest.ipinyou.com/ipinyou-dataset.pdf
- Zhang et al., *Real-Time Bidding Benchmarking with iPinYou Dataset*:
  https://arxiv.org/abs/1407.7073
- SciPy `milp` documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html
- LightGBM parameter documentation: https://lightgbm.readthedocs.io/en/latest/Parameters.html
